# Accompanying Code for "Strategies to Avoid Poor Local Minima in Nonlinear Optimization"
**Submission date**: 09.06.2026

**Author**: Nicolas Heidbrink

**Supervisor**: Stefan Schwarze, M. Sc.

**GitHub repository**: https://github.com/nicolasheidbrink/optimization-seminar


*Code structure, formatting, and documentation were optimized using Google Gemini.*

## 0. Imports

In [26]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import minimize
from contextlib import redirect_stdout
from tunneling_algorithm import TunnelingAlgorithm
import functions as fn
from algo_tester import Algorithm_Tester


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. The tunneling algorithm missing global minima
As mentioned in the seminar paper, the tunneling algorithm may not find all global minima, even for a simple objective function like $f(x) = \cos(x) + \frac15 x^2$.

Here, we find such an example and investigate the reasons by looking into the logs produced by the algorithm.


In [ ]:
i = 0 # Counter
x_stars = [np.array([-1])] * 2 # Initialization

# Search for a seed `i` that misses the global minimal point at x=-1.1253
while len(x_stars) == 2 or x_stars[0] < 0:
    i += 1
    np.random.seed(i)
    ta = TunnelingAlgorithm(fn.altered_cos, fn.altered_cos_grad, [[-10, 10]])
    x_stars, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10)]))

print(f"The seed {i} causes the algorithm to find only one minimum at x*={x_stars[0]} with an objective function value of {f_star}.")
print("The algorithm will be run again with this seed and verbose output, saving the logs to `altered_cos_logs.txt`.\n")

with open("altered_cos_logs.txt", "w") as f:
    with redirect_stdout(f):
        np.random.seed(i)
        ta = TunnelingAlgorithm(fn.altered_cos, fn.altered_cos_grad, [[-10, 10]], verbose=True)
        xx, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10)]))

The seed 78 causes the algorithm to find only one minimum at [2.12534678] with value 0.376858610099602.
The algorithm will be run again with this seed and verbose output, saving the logs to `altered_cos_logs.txt`.



An examination of the logs at lines 5 and 915 reveals that the two random perturbations attempted in the tunneling phase started at the global minimal point $x=1.1253$ were positive, pointing away from the other global minimal point at $x=-1.1253$.

The lines 1826 and 1922 further reveal that the initial points in the subsequent global search attempt were both in the area to the right of the already discovered local minimal point. The search for valid points with $T(x) \leq 0$ ended up overshooting the undiscovered global minimal point in the first case and were prevented from turning around to find it by the movable pole. In the second case, the iterates stagnated on the right side of the fixed pole at the discovered global minimal point, as described in the seminar paper.

To investigate the frequency of this issue, we run the tunneling algorithm on the function 1000 times.

In [20]:
np.random.seed(42)
failure_counter = 0
for _ in range(1000):
    ta = TunnelingAlgorithm(fn.altered_cos, fn.altered_cos_grad, [[-10, 10]])
    x_stars, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10)]))
    if len(x_stars) == 1:
        failure_counter += 1
    elif len(x_stars) not in [1, 2]:
        print(f"Unexpected number of minima found: {len(x_stars)} at points {x_stars} with value {f_star}")
        break 
print(f"The algorithm found only one of the two global minimal points {failure_counter} out of 1000 times.")

The algorithm found only one of the two global minimal points 50 out of 1000 times.


One of the global minimal points was missed 50 out of 1000 times, which just about matches the expected number of misses $1000 \times 0.5^4 = 62.5$, as explained in the seminar paper.

## 2. Improving the tunneling algorithm
To combat the problem of movable poles getting so close to tunneling phase iterates that the displacements gets extremely small, as mentioned in the seminar paper, one logical solution was to fix the movable pole's distance from iterates. Levy and Montalvo state that the movable pole should always be in the interior of a unit ball around the current iterate, so the distance was fixed to $0.9$. 

Also, a change was made to minimize the risk of random perturbations at the start of tunneling phases restricting the areas searched: These random perturbations now point generally in both directions down each axis of the domain. The changes were implemented in $\texttt{tunneling\_algorithm.py}$ and are activated by setting $\texttt{tunneling\_algorithm.improved}$ to $\texttt{True}$ while initializing the object.

The effect of the changes can be examined by testing both versions of the algorithm with $\texttt{algo\_tester.py}$, a script that runs the algorithms on a variety of functions and records whether and how many of their global minimal points were found in how much time.

In [24]:
algorithm_tester = Algorithm_Tester()
algorithm_tester.run_tests(TunnelingAlgorithm, improved=False)

Test: Shubert 2D, Avg Minima Found: 2.12 out of 18, Failure Rate: 20.00% over 40 trials, Time per Trial: 0.386 seconds.
Test: Shubert 3D, Avg Minima Found: 1.50 out of 81, Failure Rate: 80.00% over 10 trials, Time per Trial: 0.473 seconds.
Test: Shubert 6D, Avg Minima Found: 0.00 out of 4374, Failure Rate: 100.00% over 10 trials, Time per Trial: 0.800 seconds.
Test: Six Hump Camel, Avg Minima Found: 1.80 out of 2, Failure Rate: 0.00% over 30 trials, Time per Trial: 0.092 seconds.
Test: Altered Shubert 2D, Avg Minima Found: 1.00 out of 1, Failure Rate: 83.33% over 30 trials, Time per Trial: 0.319 seconds.
Test: Function 15 2D, Avg Minima Found: 1.00 out of 1, Failure Rate: 3.33% over 30 trials, Time per Trial: 0.171 seconds.
Test: Function 15 3D, Avg Minima Found: 1.00 out of 1, Failure Rate: 20.00% over 20 trials, Time per Trial: 0.245 seconds.
Test: Function 15 4D, Avg Minima Found: 1.00 out of 1, Failure Rate: 60.00% over 10 trials, Time per Trial: 0.328 seconds.
Test: Function 16 5D

In [25]:
algorithm_tester.run_tests(TunnelingAlgorithm, improved=True)

Test: Shubert 2D, average number of minima found when successful: 13.95 out of 18, failure rate: 0.00% over 40 trials, time per trial: 1.373 seconds.
Test: Shubert 3D, average number of minima found when successful: 2.70 out of 81, failure rate: 0.00% over 10 trials, time per trial: 0.323 seconds.
Test: Shubert 6D, average number of minima found when successful: 1.00 out of 4374, failure rate: 80.00% over 10 trials, time per trial: 0.241 seconds.
Test: Six Hump Camel, average number of minima found when successful: 2.00 out of 2, failure rate: 0.00% over 30 trials, time per trial: 0.106 seconds.
Test: Altered Shubert 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 20.00% over 30 trials, time per trial: 0.329 seconds.
Test: Function 15 2D, average number of minima found when successful: 1.00 out of 1, failure rate: 0.00% over 30 trials, time per trial: 0.188 seconds.
Test: Function 15 3D, average number of minima found when successful: 1.00 out of 1, fai

Clearly, the results with the improvements are superior, although they still do not match Levy and Montalvo's results.

In [11]:
algorithm_tester = Algorithm_Tester()
algorithm_tester.run_tests(TunnelingAlgorithm, verbose=False, improved=True, annealing=True)

c:\Users\nicol\DS_Projects\Seminar_paper\python\algorithms.py:11: RuntimeWarning: divide by zero encountered in scalar divide
  self.lambda_list = [] # Pole strengths for each x_star


12 minima found with value -186.7309088310238
16 minima found with value -186.73090883102387
14 minima found with value -186.73090883102384
18 minima found with value -186.73090883101275
18 minima found with value -186.73090883102373
18 minima found with value -186.7309088310238
18 minima found with value -186.7309088310207
11 minima found with value -186.7309088309921
16 minima found with value -186.730908831023
16 minima found with value -186.73090883102384
13 minima found with value -186.73090883102358
16 minima found with value -186.7309088310238
18 minima found with value -186.7309088310238
13 minima found with value -186.73090883102356
17 minima found with value -186.73090883101472
18 minima found with value -186.73090883102356
17 minima found with value -186.7309088310224
18 minima found with value -186.73090883100818
17 minima found with value -186.73090883102378
14 minima found with value -186.73090883102333
17 minima found with value -186.7309088310237
18 minima found with va

In [ ]:
for _ in range(50):
    ta = TunnelingAlgorithm(fn.shubert, fn.shubert_grad, [[-10, 10], [-10, 10]], verbose=False, improved=True, annealing=True)
    with open("logs.txt", "w") as f:
        with redirect_stdout(f):
            xx, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10), np.random.uniform(-10, 10)]))
            print(xx)
    print(f"Annealing found {len(xx)} minima with value {f_star} at {xx}")
for _ in range(50):
    ta = TunnelingAlgorithm(fn.shubert, fn.shubert_grad, [[-10, 10], [-10, 10]], verbose=False, improved=True, annealing=False)
    with open("logs.txt", "w") as f:
        with redirect_stdout(f):
            xx, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10), np.random.uniform(-10, 10)]))
            print(xx)
    print(f"Regular found {len(xx)} minima with value {f_star} at {xx}")

In [ ]:
ta = TunnelingAlgorithm(fn.shubert, fn.shubert_grad, [[-10, 10], [-10, 10]], verbose=True, improved=True, annealing=True)
with open("logs.txt", "w") as f:
    with redirect_stdout(f):
        xx, f_star = ta.apply_algorithm(np.array([np.random.uniform(-10, 10), np.random.uniform(-10, 10)]))
        print(xx)
print(f"Annealing found {len(xx)} minima with value {f_star} at {xx}")